In [2]:
import random, uuid

customers_int = 10
ts_step_int = 1

class OrderGenerator:
    def __init__(self):
        self.customer_id_int = 0
        #
        self.ts_int = 0
        self.ts_step_int = ts_step_int

    def generate(self):
        order_id_str = str(uuid.uuid4())
        message_dict = {
            "key": order_id_str,
            "value": {"order_id": order_id_str,
                      "customer_id": random.randint(0, customers_int - 1),
                      "price": random.randint(1, 10000) / 100,
                      "ts": self.ts_int},
        }
        #
        self.ts_int += self.ts_step_int
        #
        return message_dict


In [376]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

#

order_source_str = "orders"
sink_str = "orders_tumbling"
#
tumbling_int = ts_step_int * 100
allowed_lateness_intervals = 1
allowed_lateness_ms = allowed_lateness_intervals * tumbling_int
#
order_tn = (
    Tn.source(order_source_str)
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .expire(lambda x: x["ts"],
            lambda x: (x // tumbling_int) * tumbling_int + tumbling_int + allowed_lateness_ms)
    .distinct()
)
#
group_by_tn = (
    order_tn
    .group_by_agg(
        by_function=lambda x: (
            (x["ts"] // tumbling_int) * tumbling_int + tumbling_int,
            x["customer_id"]
        ),
        select_function=lambda x: x,
        projection_function=lambda by, agg: {
            "window_start": by[0] - tumbling_int,
            "window_end": by[0],
            "customer_id": by[1],
            "orders": agg["orders"],
            "total_price": agg["total_price"],
            "last_ts": agg["last_ts"]
        },
        agg_function=lambda agg, x, w: {
            "orders": agg["orders"] + w,
            "total_price": agg["total_price"] + x["price"] * w,
            "last_ts": max(agg["last_ts"], x["ts"]) if w > 0 else agg["last_ts"]
        },
        agg_initial_any={
            "orders": 0,
            "total_price": 0,
            "last_ts": 0
        }
    )
)
#
def get_expired_tumbling_windows(max_ts):
    current_closed_end = (max_ts // tumbling_int) * tumbling_int
    return [
        {"window_end": current_closed_end - i * tumbling_int}
        for i in range(allowed_lateness_intervals + 1)
        if current_closed_end - i * tumbling_int >= tumbling_int
    ]

window_tn = (
    group_by_tn
    .join_equi(
        order_tn
        .map(lambda x: x["ts"])
        .max(lambda x: x)
        .flatmap(lambda max_ts: get_expired_tumbling_windows(max_ts)),
        lambda l: l["window_end"],
        lambda r: r["window_end"],
        lambda l, _: l
    )
)._peek("expel")
#
built_tn = Tn.build(window_tn.sink(sink_str))



In [377]:
built_tn.reset()

def push(customer_id, price, ts):
    built_tn.push(order_source_str, [{'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}])
    built_tn.latest()

# tumbling_int=100

print("=== Window [0,100) running ===")
push(1, 100, 10)  # window_end=100
push(1, 200, 50)  # window_end=100, same bucket
push(2, 150, 70)  # window_end=100, customer 2

print("\n=== Window [0,100) closing → Emit ===")
push(1, 300, 105) # watermark=105 > 100 → both customers fire!
                  # customer 1: orders=2, total=300
                  # customer 2: orders=1, total=150

print("\n=== Window [100,200) running ===")
push(2, 200, 120) # window_end=200
push(1, 100, 150) # window_end=200
push(2, 300, 180) # window_end=200

print("\n=== Late Arrival for window [0,100) – still inside allowed lateness ===")
push(1, 500, 90)  # ts=90 → window_end=100, watermark=180 < 200 → correction!
                  # customer 1: +{orders=3, total=800} / -{orders=2, total=300}

print("\n=== Window [100,200) closing → Emit ===")
push(1, 100, 205) # watermark=205 > 200 → both customers fire!
                  # customer 1: orders=2, total=400
                  # customer 2: orders=2, total=500

=== Window [0,100) running ===

=== Window [0,100) closing → Emit ===
expel: ({'window_start': 0, 'window_end': 100, 'customer_id': 1, 'orders': 2, 'total_price': 300, 'last_ts': 50}, 1)
expel: ({'window_start': 0, 'window_end': 100, 'customer_id': 2, 'orders': 1, 'total_price': 150, 'last_ts': 70}, 1)

=== Window [100,200) running ===

=== Late Arrival for window [0,100) – still inside allowed lateness ===
expel: ({'window_start': 0, 'window_end': 100, 'customer_id': 1, 'orders': 2, 'total_price': 300, 'last_ts': 50}, -1)
expel: ({'window_start': 0, 'window_end': 100, 'customer_id': 1, 'orders': 3, 'total_price': 800, 'last_ts': 90}, 1)

=== Window [100,200) closing → Emit ===
expel: ({'window_start': 100, 'window_end': 200, 'customer_id': 2, 'orders': 2, 'total_price': 500, 'last_ts': 180}, 1)
expel: ({'window_start': 100, 'window_end': 200, 'customer_id': 1, 'orders': 2, 'total_price': 400, 'last_ts': 150}, 1)
expel: ({'window_start': 0, 'window_end': 100, 'customer_id': 2, 'orders'

In [7]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_hopping"
#
tumbling_int = ts_step_int * 100
hop_int = tumbling_int // 2  # hop=50, window=100 → jedes Event in 2 Buckets
# Wie viele Hops dürfen Events maximal zu spät ankommen?
allowed_lateness_hops = 3
# Das entspricht in Sekunden/Millisekunden:
allowed_lateness_ms = allowed_lateness_hops * hop_int  # 2 * 50 = 100
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .expire(lambda x: x["ts"],
            lambda x: (x // tumbling_int) * tumbling_int + tumbling_int + allowed_lateness_ms)
    .distinct()
)
#
def assign(ts):
    first_end = (ts // hop_int) * hop_int + hop_int
    x = [
        first_end + i * hop_int 
        for i in range(tumbling_int // hop_int)
        if first_end + i * hop_int >= tumbling_int
    ]
    print(f"assign: ts: {ts}, windows: {x}")
    return x
#
# 1. Oben: Jedes Event in seine Hops duplizieren (Flatmap)
group_by_tn = (
    order_tn
    .flatmap(lambda x: [
        {**x, "ts_orig": x["ts"], "window_end_hint": we}
        for we in assign(x["ts"])
    ])
    .group_by_agg(
        by_function=lambda x: (x["window_end_hint"], x["customer_id"]),
        select_function=lambda x: x,
        projection_function=lambda by, agg: {
            "window_start": by[0] - tumbling_int,
            "window_end": by[0],
            "customer_id": by[1],
            "orders": agg["orders"],
            "total_price": agg["total_price"]
        },
        # Super simpler, performanter State!
        agg_function=lambda agg, x, w: {
            "orders": agg["orders"] + w,
            "total_price": agg["total_price"] + x["price"] * w,
        },
        agg_initial_any={"orders": 0, "total_price": 0}
    )
)

# 2. Unten: Der Gatekeeper lässt das Fenster erst durch, wenn die Zeit reif ist
window_tn = (
    group_by_tn
    .join(
        order_tn
        .map(lambda x: x["ts"])
        .max(lambda x: x),  # Einfach nur die globale Uhr (max_ts) holen
        # Das Fenster passiert das Tor, sobald max_ts die Lateness-Grenze überschreitet
        lambda l, r: r >= l["window_end"],
        lambda l, _: l
    )
    ._filter(lambda record, w: w > 0) # Retractions blockieren
)._peek("emit")
#
built_tn = Tn.build(window_tn.sink(sink_str))

In [8]:
built_tn.reset()

def push(customer_id, price, ts, w=1):
    built_tn.push(order_source_str, [({'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}, w)])
    built_tn.latest()

# tumbling=100, hop=50 → windows: [0,100), [50,150), [100,200), ...
# ts=10  → window_ends: 100 (only [0,100))
# ts=60  → window_ends: 100, 150 (in [0,100) AND [50,150)!)
# ts=105 → window_ends: 150, 200 (in [50,150) AND [100,200)!)

print("=== Events in window [0,100) ===")
push(1, 100, 10)  # → only window_end=100
push(1, 200, 60)  # → window_end=100 AND 150

print("\n=== [0,100) closes → Emit ===")
push(1, 300, 105) # watermark=105 >     100
                  # → [0,100): orders=2, total=300 (ts=10 + ts=60)
                  # ts=105 goes into [50,150) AND [100,200)

print("\n=== [50,150) closes → Emit ===")
push(1, 400, 155) # watermark=155 > 150
                  # → [50,150): orders=2, total=500 (ts=60 + ts=105)

print("\n=== [100,200) closes → Emit ===")
push(1, 100, 205) # watermark=205 > 200
                  # → [100,200): orders=2, total=700 (ts=105 + ts=155)

print("\n=== late arrivals ===")
push(1, 42, 49)
push(1, 23, 101)

print("\n=== [200,300) closes → Emit ===")
push(1, 100, 305) # watermark=305 > 300
                  # → [200,300): orders=2, total=700 (ts=105 + ts=155)

print("\n=== late arrival ===")
push(1, 4711, 101)

print("\n=== too late arrival ===")
push(1, 4711, 49)



=== Events in window [0,100) ===
assign: ts: 10, windows: [100]
assign: ts: 60, windows: [100, 150]

=== [0,100) closes → Emit ===
assign: ts: 105, windows: [150, 200]
emit: ({'window_start': 0, 'window_end': 100, 'customer_id': 1, 'orders': 2, 'total_price': 300}, 1)

=== [50,150) closes → Emit ===
assign: ts: 155, windows: [200, 250]
emit: ({'window_start': 50, 'window_end': 150, 'customer_id': 1, 'orders': 2, 'total_price': 500}, 1)

=== [100,200) closes → Emit ===
assign: ts: 205, windows: [250, 300]
emit: ({'window_start': 100, 'window_end': 200, 'customer_id': 1, 'orders': 2, 'total_price': 700}, 1)

=== late arrivals ===
assign: ts: 49, windows: [100]
emit: ({'window_start': 0, 'window_end': 100, 'customer_id': 1, 'orders': 3, 'total_price': 342}, 1)
assign: ts: 101, windows: [150, 200]
emit: ({'window_start': 50, 'window_end': 150, 'customer_id': 1, 'orders': 3, 'total_price': 523}, 1)
emit: ({'window_start': 100, 'window_end': 200, 'customer_id': 1, 'orders': 3, 'total_price':

In [9]:
built_tn.reset()

def push(customer_id, price, ts, w=1):
    built_tn.push(order_source_str, [({'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}, w)])
    built_tn.latest()

# Konfiguration im Hinterkopf: tumbling_int=100, hop_int=50
# allowed_lateness_ms = 150
# Fenster 100: [0, 100)
# Fenster 150: [50, 150)
# Fenster 200: [100, 200)

print("=== 1. Normale Events für das erste Fenster ===")
push(1, 100, 10)  # In Fenster 100
push(1, 200, 60)  # In Fenster 100 UND Fenster 150
# Erwartung: Kein Output (Gatekeeper blockiert, max_ts = 60)

print("\n=== 2. Trigger Event -> Schließt Fenster 100 ===")
push(1, 300, 105) # max_ts = 105 -> Schließt Fenster 100 (da 105 >= 100)
                  # In Fenster 150 UND Fenster 200
# Erwartung: Emit für Fenster 100 (orders: 2, price: 300 [10 + 60])

print("\n=== 3. Erlaubte Lateness (In-Order für Fenster 150, aber LATE für Fenster 100) ===")
# Fenster 100 ist schon emittiert, aber der State ist wegen .expire() noch offen!
push(1, 50, 40)   # Fällt in Fenster 100
# Erwartung: Sofortiges Delta-Update/Korrektur für das bereits geschlossene Fenster 100!
# Output Fenster 100: orders: 3, price: 350

print("\n=== 4. Trigger Event -> Schließt Fenster 150 ===")
push(1, 400, 155) # max_ts = 155 -> Schließt Fenster 150 (155 >= 150)
                  # In Fenster 200 UND Fenster 250
# Erwartung: Emit für Fenster 150 (orders: 3, price: 550 [60 + 105 + 400])

print("\n=== 5. Native Retraction (Kunde storniert ein Event) ===")
# Wir ziehen das Event von vorhin (ts=105, price=300) mit Gewicht w=-1 ab!
push(1, 300, 105, w=-1)
# Erwartung: Da Fenster 150 und 200 betroffen sind:
# - Fenster 150 (bereits offen) emittiert ein Korrektur-Update (total_price sinkt um 300)
# - Im Zustand für Fenster 200 wird es direkt abgezogen (bevor es je emittiert wurde!)

print("\n=== 6. Hart an der Grenze: Letzte erlaubte Lateness ===")
# Für Fenster 100 ist der Expire-Cutoff bei ts=250. Wir senden ts=30 (gehört in 100)
push(1, 99, 30)
# Erwartung: Da max_ts noch 155 ist, schlägt das .expire() noch nicht zu.
# Fenster 100 kriegt ein weiteres Update!

print("\n=== 7. Der große Uhr-Sprung -> Auslösung & Expire-Grenze überschreiten ===")
push(1, 100, 255) # max_ts springt auf 255!
                  # Schließt Fenster 200 & 250.
                  # Löst zeitgleich das .expire() für alle Events < 100 aus!
# Erwartung: 
# - Emit für Fenster 200 (ohne das stornierte ts=105 Event!)
# - Der State säubert sich im Hintergrund für die alten Epochen.

print("\n=== 8. TOO LATE ARRIVAL (Gnadenlos verworfen) ===")
# Wir versuchen JETZT nochmal ein Event für das allererste Fenster zu senden
push(1, 999, 20)  # Gehört in Fenster 100
# Erwartung: KEIN Output. Das Event wird vom .expire() oben im Graphen gefiltert,
# da max_ts (255) > 20 + 100 + 150 (Cutoff) ist. Der Zustand existiert nicht mehr.

=== 1. Normale Events für das erste Fenster ===
assign: ts: 10, windows: [100]
assign: ts: 60, windows: [100, 150]

=== 2. Trigger Event -> Schließt Fenster 100 ===
assign: ts: 105, windows: [150, 200]
emit: ({'window_start': 0, 'window_end': 100, 'customer_id': 1, 'orders': 2, 'total_price': 300}, 1)

=== 3. Erlaubte Lateness (In-Order für Fenster 150, aber LATE für Fenster 100) ===
assign: ts: 40, windows: [100]
emit: ({'window_start': 0, 'window_end': 100, 'customer_id': 1, 'orders': 3, 'total_price': 350}, 1)

=== 4. Trigger Event -> Schließt Fenster 150 ===
assign: ts: 155, windows: [200, 250]
emit: ({'window_start': 50, 'window_end': 150, 'customer_id': 1, 'orders': 2, 'total_price': 500}, 1)

=== 5. Native Retraction (Kunde storniert ein Event) ===
assign: ts: 105, windows: [150, 200]
emit: ({'window_start': 50, 'window_end': 150, 'customer_id': 1, 'orders': 1, 'total_price': 200}, 1)

=== 6. Hart an der Grenze: Letzte erlaubte Lateness ===
assign: ts: 30, windows: [100]
emit: (

In [300]:
built_tn.reset()

def push(customer_id, price, ts):
    built_tn.push(order_source_str, [{'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}])
    res = built_tn.latest()
    if res:
        print(f"Output bei ts={ts}: {res}")

# tumbling=100, hop=50 → windows: [0,100), [50,150), [100,200), ...

print("=== Normaler Anlauf ===")
push(1, 100, 10)   # Fenster-Ende: 100
push(1, 200, 60)   # Fenster-Ende: 100, 150

print("\n=== [0,100) schließt ===")
push(1, 300, 105)  # watermark=105 -> schließt [0,100)

print("\n=== DER ZEITSPRUNG (Simuliert Lücke / Batch-Schub) ===")
# Dieses Event zieht max_ts schlagartig auf 500 hoch.
# triggers(500) liefert nur noch Fenster ab 350+ (500 - 3 * 50).
# Das Fenster für 450/500 wird geöffnet.
push(1, 100, 500)  

print("\n=== ERLAUBTE LATE ARRIVAL (Sollte durchgehen) ===")
# ts=410 ist vollkommen valide (Latenz erlaubt bis zu ~150ms Verspätung)
# assign(410) packt es korrekt in die Fenster-Enden 450 und 500.
# ABER: Da max_ts bei 500 bleibt, feuert der Max-Stream kein Update.
# Da das Fenster 450 nicht mehr in triggers(500) aktiv im Join-Zustand gehalten wird,
# verschwindet dieses Update im Nichts!
push(1, 50, 410)  

print("\n=== TOO LATE ARRIVAL (Sollte gefiltert werden) ===")
# ts=100 ist viel zu spät bei max_ts=500 -> fliegt flach durch .expire() raus.
push(1, 999, 100)

=== Normaler Anlauf ===
assign: ts: 10, windows: [100]
triggers: current_window_end: 0, windows: []
assign: ts: 60, windows: [100, 150]
triggers: current_window_end: 50, windows: []
triggers: current_window_end: 0, windows: []

=== [0,100) schließt ===
assign: ts: 105, windows: [150, 200]
triggers: current_window_end: 100, windows: [100]
triggers: current_window_end: 50, windows: []
emit: ({'window_start': 0, 'window_end': 100, 'customer_id': 1, 'orders': 2, 'total_price': 300, 'last_ts': 60}, 1)
Output bei ts=105: {'orders_hopping': [{'window_start': 0, 'window_end': 100, 'customer_id': 1, 'orders': 2, 'total_price': 300, 'last_ts': 60}]}

=== DER ZEITSPRUNG (Simuliert Lücke / Batch-Schub) ===
assign: ts: 500, windows: [550, 600]
assign: ts: 105, windows: [150, 200]
assign: ts: 60, windows: [100, 150]
assign: ts: 10, windows: [100]
triggers: current_window_end: 500, windows: [500, 450, 400, 350]
triggers: current_window_end: 100, windows: [100]
emit: ({'window_start': 0, 'window_end':

In [8]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_cumulative"
#
max_window_size = ts_step_int * 100  # Das "Großfenster" (z.B. ein ganzer Tag / 100)
step_int = max_window_size // 5      # Die Schrittweite (z.B. jede Stunde / 20)
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .expire(lambda x: x["ts"],
            lambda x: (x // max_window_size) * max_window_size + max_window_size)
    .distinct()
)
#

def assign_cumulative(ts):
    # Finde den Start des aktuellen Großfensters (z.B. Mitternacht)
    macro_start = (ts // max_window_size) * max_window_size
    
    # Finde das erste Schritt-Ende nach dem Event
    first_step_end = ((ts // step_int) * step_int) + step_int
    
    # Das Ende des gesamten Großfensters
    macro_end = macro_start + max_window_size
    
    # Das Event gehört in das aktuelle Teilfenster UND alle zukünftigen Teilfenster dieses Tages
    x = [
        end for end in range(first_step_end, macro_end + step_int, step_int)
        if end <= macro_end
    ]
    print(f"assign_cumulative: ts: {ts}, windows: {x}")
    return x
#
# 1. Oben: Jedes Event in seine Hops duplizieren (Flatmap)
group_by_tn = (
    order_tn
    .flatmap(lambda x: [
        {**x, "ts_orig": x["ts"], "window_end_hint": we}
        for we in assign_cumulative(x["ts"])
    ])
    .group_by_agg(
        by_function=lambda x: (x["window_end_hint"], x["customer_id"]),
        select_function=lambda x: x,
        projection_function=lambda by, agg: {
            "window_end": by[0],
            "customer_id": by[1],
            "orders": agg["orders"],
            "total_price": agg["total_price"]
        },
        # Super simpler, performanter State!
        agg_function=lambda agg, x, w: {
            "orders": agg["orders"] + w,
            "total_price": agg["total_price"] + x["price"] * w,
        },
        agg_initial_any={"orders": 0, "total_price": 0}
    )
)

# 2. Unten: Der Gatekeeper lässt das Fenster erst durch, wenn die Zeit reif ist
window_tn = (
    group_by_tn
    .join(
        order_tn
        .map(lambda x: x["ts"])
        .max(lambda x: x),  # Einfach nur die globale Uhr (max_ts) holen
        # Das Fenster passiert das Tor, sobald max_ts die Lateness-Grenze überschreitet
        lambda l, r: r >= l["window_end"],
        lambda l, _: l
    )
    ._filter(lambda record, w: w > 0) # Retractions blockieren
)._peek("emit")
#
built_tn = Tn.build(window_tn.sink(sink_str))


In [10]:
built_tn.reset()

def push(customer_id, price, ts, w=1):
    built_tn.push(order_source_str, [({'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}, w)])
    built_tn.latest()

print("=== 1. Erstes Event im ersten Slice ===")
push(1, 100, 10)  
# assign_cumulative -> [20, 40, 60, 80, 100]
# Erwartung: Kein Output (max_ts = 10 < 20)

print("\n=== 2. Event im zweiten Slice -> Löst das 20er-Fenster aus ===")
push(1, 50, 25)   
# assign_cumulative -> [40, 60, 80, 100]
# max_ts = 25 -> Löst Fenster 20 aus!
# Erwartung: Emit für window_end=20 (orders: 1, total_price: 100 [nur das ts=10 Event])

print("\n=== 3. Weiteres Event im selben Makro-Fenster -> Löst das 40er-Fenster aus ===")
push(1, 200, 45)  
# assign_cumulative -> [60, 80, 100]
# max_ts = 45 -> Löst Fenster 40 aus!
# Erwartung: Emit für window_end=40 (orders: 2, total_price: 150 [ts=10 + ts=25])

print("\n=== 4. Late Arrival (Kommt bei max_ts=45 an, gehört aber zu ts=15) ===")
# Das Event landet rückwirkend in [20, 40, 60, 80, 100]
push(1, 10, 15)   
# Erwartung: Sofortige Delta-Updates für die bereits emittierten Fenster 20 und 40!
# - Emit für window_end=20: orders: 2, total_price: 110
# - Emit für window_end=40: orders: 3, total_price: 160

print("\n=== 5. Native Retraction (Storno eines alten Events während des Tages) ===")
# Wir stornieren das Event von ts=25 (Wert 50) mit w=-1
push(1, 50, 25, w=-1)
# Erwartung: Korrektur-Updates für alle betroffenen Fenster, in denen es kumuliert war!
# - Fenster 20 ist NICHT betroffen (da ts=25 dort nie drin war).
# - Fenster 40 zieht die 50 ab -> orders: 2, total_price: 110.
# - In den noch offenen Zuständen (60, 80, 100) korrigiert es sich lautlos im Vorfeld.

print("\n=== 6. Großer Zeitsprung -> Löst kaskadenartig restliche Fenster aus ===")
push(1, 500, 95)  
# assign_cumulative -> [100]
# max_ts springt auf 95 -> Löst Fenster 60 und 80 zeitgleich aus!
# Erwartung:
# - Emit für window_end=60 (Kumulierter Wert bis ts=60 ohne das stornierte Event)
# - Emit für window_end=80 (Gleicher kumulierter Wert, da zwischen 60 und 80 nichts passierte)

print("\n=== 7. Sprung ins NÄCHSTE Makro-Fenster -> Schließt das alte Großfenster & triggert Expire ===")
push(1, 1000, 105) 
# assign_cumulative -> [120, 140, 160, 180, 200] (Neuer Tag!)
# max_ts = 105 -> Schließt das finale Fenster 100 des Vortrags ab.
# Zeitgleich greift das .expire() für alle Daten des ersten Tages (< 100).
# Erwartung: 
# - Emit für window_end=100 (Inhalt: Alle gültigen Tagesumsätze inklusive des ts=95er Events)
# - Im Hintergrund säubert DBSP den Zustand des ersten Tages restlos.

print("\n=== 8. TOO LATE ARRIVAL (Vortag ist abgelaufen) ===")
push(1, 999, 45)  
# Erwartung: Kein Output. Das Event wird oben durch .expire() direkt verworfen, 
# da das Makro-Fenster 0-100 im Graphen bereits gelöscht wurde.


=== 1. Erstes Event im ersten Slice ===
assign_cumulative: ts: 10, windows: [20, 40, 60, 80, 100]

=== 2. Event im zweiten Slice -> Löst das 20er-Fenster aus ===
assign_cumulative: ts: 25, windows: [40, 60, 80, 100]
emit: ({'window_end': 20, 'customer_id': 1, 'orders': 1, 'total_price': 100}, 1)

=== 3. Weiteres Event im selben Makro-Fenster -> Löst das 40er-Fenster aus ===
assign_cumulative: ts: 45, windows: [60, 80, 100]
emit: ({'window_end': 40, 'customer_id': 1, 'orders': 2, 'total_price': 150}, 1)

=== 4. Late Arrival (Kommt bei max_ts=45 an, gehört aber zu ts=15) ===
assign_cumulative: ts: 15, windows: [20, 40, 60, 80, 100]
emit: ({'window_end': 20, 'customer_id': 1, 'orders': 2, 'total_price': 110}, 1)
emit: ({'window_end': 40, 'customer_id': 1, 'orders': 3, 'total_price': 160}, 1)

=== 5. Native Retraction (Storno eines alten Events während des Tages) ===
assign_cumulative: ts: 25, windows: [40, 60, 80, 100]
emit: ({'window_end': 40, 'customer_id': 1, 'orders': 2, 'total_price'

In [ ]:
built_tn.reset()

def push(customer_id, price, ts):
    built_tn.push(order_source_str, [{'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}])
    built_tn.latest()

# max_size=100, step=25 → windows [0,25), [0,50), [0,75), [0,100) für jeden Customer

print("=== Erst-Event, geht in alle 4 Buckets ===")
push(1, 100, 10)  # → window_ends: 25, 50, 75, 100

print("\n=== Step 25 überschritten → [0,25) emittet ===")
push(1, 200, 30)  # watermark=30 > 25 → [0,25) feuert mit orders=1, total=100

print("\n=== Step 50 überschritten → [0,50) emittet ===")
push(1, 300, 55)  # watermark=55 > 50 → [0,50) feuert mit orders=2, total=300

print("\n=== Step 75 überschritten → [0,75) emittet ===")
push(1, 400, 80)  # watermark=80 > 75 → [0,75) feuert mit orders=3, total=600

print("\n=== max_size überschritten → [0,100) emittet, neuer Zyklus ===")
push(1, 500, 105) # watermark=105 > 100 → [0,100) feuert mit orders=4, total=1000
                  # ts=105 startet neuen Zyklus: window_ends 125, 150, 175, 200

In [506]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_session"

ts_step_int = 1
gap_int = ts_step_int * 20
max_session_int = ts_step_int * 200
lateness_int = 0

zero = {"orders": 0, "price": 0}
combine = lambda a, b: {"orders": a["orders"] + b["orders"], "price": a["price"] + b["price"]}
event_value = lambda x: {"orders": 1, "price": x["price"]}

def merge_sessions(raw_events, gap_int):
    sessions = []
    for ts, value in sorted(raw_events):
        if not sessions or ts - sessions[-1]["last_ts"] > gap_int:
            sessions.append({"last_ts": ts, "value": zero})
        sessions[-1] = {"last_ts": ts, "value": combine(sessions[-1]["value"], value)}
    return [{**s, "window_end": s["last_ts"] + gap_int} for s in sessions]

def bla(agg, x, w):
    ts_map = dict(agg.get("raw_events", []))
    ts_map[x["ts"]] = combine(ts_map.get(x["ts"], zero), event_value(x))
    raw_events = list(ts_map.items())
    #
    sessions = merge_sessions(raw_events, gap_int)
    last = sessions[-1] if sessions else {"window_end": 0, "value": zero}
    #
    return {
        "window_end": last["window_end"],
        "orders": last["value"]["orders"],
        "total_price": last["value"]["price"],
        "sessions": [{"window_end": s["window_end"], **s["value"]} for s in sessions],
        "raw_events": raw_events
    }

order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .expire(lambda x: x["ts"],
            lambda x: ((x // max_session_int) * max_session_int + max_session_int) + lateness_int)
    .distinct()
)

#
# 2. Gruppierung und Flatmap
#
group_by_tn = (
    order_tn
    .group_by_agg(
        by_function=lambda x: x["customer_id"],
        select_function=lambda x: x,
        projection_function=lambda by, agg: {
            "customer_id": by,
            "window_end": agg["window_end"],
            "orders": agg["orders"],
            "total_price": agg["total_price"],
            "sessions": agg["sessions"]
        },
        agg_function=bla,
        agg_initial_any={"window_end": 0, "orders": 0, "total_price": 0, "sessions": [], "raw_events": []}
    )
).flatmap(lambda x: [
        {
            "customer_id": x["customer_id"],
            "window_end": s["window_end"],
            "orders": s["orders"],
            "total_price": s["price"]
        } for s in x["sessions"]
    ])

#
# 3. Gatekeeper-Join für Lateness
#
window_tn = (
    group_by_tn
    .join(
        order_tn
        .map(lambda x: x["ts"])
        .max(lambda x: x, lambda x: x),
        lambda l, r: r >= l["window_end"] + lateness_int,
        lambda l, _: l
    )
    ._filter(lambda record, w: w > 0)
)._peek("emit")

#
# 4. Build
#
built_tn = Tn.build(window_tn.sink(sink_str))


In [531]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

order_source_str = "orders"
sink_str = "orders_session"

ts_step_int = 1
gap_int = ts_step_int * 20
max_session_int = ts_step_int * 200
lateness_int = 0

zero = {"orders": 0, "price": 0}
combine = lambda a, b: {"orders": a["orders"] + b["orders"], "price": a["price"] + b["price"]}
event_value = lambda x: {"orders": 1, "price": x["price"]}

import bisect

def insert_session(sessions, ts, value, gap_int):
    """sessions: sortierte Liste [{'start','last_ts','value'}, ...].
       Reiner Insert-Fall (w immer 1) -> O(log n) Suche + O(1) Merge, kein Full-Rebuild."""
    starts = [s["start"] for s in sessions]
    i = bisect.bisect_right(starts, ts) - 1
    #
    # ts liegt bereits innerhalb einer bestehenden Session (gleicher ts nochmal aggregiert)
    if i >= 0 and sessions[i]["start"] <= ts <= sessions[i]["last_ts"]:
        sessions[i]["value"] = combine(sessions[i]["value"], value)
        return sessions
    #
    left = sessions[i] if i >= 0 else None
    right = sessions[i+1] if i+1 < len(sessions) else None
    merges_left = left is not None and ts - left["last_ts"] <= gap_int
    merges_right = right is not None and right["start"] - ts <= gap_int
    #
    if merges_left and merges_right:
        merged = {"start": left["start"], "last_ts": right["last_ts"],
                  "value": combine(combine(left["value"], value), right["value"])}
        sessions[i:i+2] = [merged]
    elif merges_left:
        left["last_ts"] = ts
        left["value"] = combine(left["value"], value)
    elif merges_right:
        right["start"] = ts
        right["value"] = combine(value, right["value"])
    else:
        sessions.insert(i+1, {"start": ts, "last_ts": ts, "value": value})
    #
    return sessions

order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .expire(lambda x: x["ts"],
            lambda x: ((x // max_session_int) * max_session_int + max_session_int) + lateness_int)
    .distinct()
)

group_by_tn = (
    order_tn
    .group_by_agg(
        by_function=lambda x: x["customer_id"],
        select_function=lambda x: x,
        projection_function=lambda by, agg: {
            "customer_id": by,
            "output": agg["output"]
        },
        agg_function=lambda agg, x, _: {
        "sessions": (sessions := insert_session(agg.get("sessions", []), x["ts"], event_value(x), gap_int)),
        "output": [{"window_end": s["last_ts"] + gap_int, **s["value"]} for s in sessions]
        },
        agg_initial_any={"sessions": [], "output": []}
    )
).flatmap(lambda x: [
        {
            "customer_id": x["customer_id"],
            "window_end": s["window_end"],
            "orders": s["orders"],
            "total_price": s["price"]
        } for s in x["output"]
    ])

window_tn = (
    group_by_tn
    .join(
        order_tn
        .map(lambda x: x["ts"])
        .max(lambda x: x, lambda x: x),
        lambda l, r: r >= l["window_end"] + lateness_int,
        lambda l, _: l
    )
)._peek("emit")

built_tn = Tn.build(window_tn.sink(sink_str))


In [532]:
built_tn.reset()

def push(customer_id, price, ts, w=1):
    built_tn.push(order_source_str, [({'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}, w)])
    built_tn.latest()

#

push(1, 100, 5, 1)
push(1, 100, 5, -1)
push(1, 100, 15, 1)
push(1, 100, 35, 1)
push(1, 100, 36, 1)
push(1, 100, 57, 1)


emit: ({'customer_id': 1, 'window_end': 56, 'orders': 3, 'total_price': 300}, 1)


In [533]:
built_tn.reset()

def push(customer_id, price, ts, w=1):
    built_tn.push(order_source_str, [({'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}, w)])
    built_tn.latest()

# gap=20, max_session=200

print("=== Session customer 1 läuft ===")
push(1, 100, 0)   # session_bucket=0
push(1, 200, 10)  # session_bucket=0, verlängert
push(1, 300, 15)  # session_bucket=0, verlängert

print("\n=== Session customer 2 läuft parallel ===")
push(2, 150, 5)   # session_bucket=0
push(2, 250, 18)  # session_bucket=0, verlängert

print("\n=== Gap customer 1 abgelaufen → Emit ===")
push(2, 100, 60)  # watermark=60 > 15+20=35 → customers 1 feuert! orders=3, total=600, und customer 2 auch: orders: 2, total 400

print("\n=== Gap customer 2 abgelaufen → Emit ===")
push(1, 100, 90)  # watermark=90 > 18+20=38 → customer 2 feuert! orders=1, total=100

print("\n=== Hard Reset bei max_session_int → neue session_bucket ===")
push(1, 500, 205) # session_bucket=200 → frischer state!
push(1, 500, 210) # session_bucket=200, verlängert

print("\n=== Gap neue Session customer 1 abgelaufen → Emit ===")
push(2, 50, 250)  # watermark=250 > 210+20=230 → customer 1 feuert! orders=2, total=1000

=== Session customer 1 läuft ===

=== Session customer 2 läuft parallel ===

=== Gap customer 1 abgelaufen → Emit ===
emit: ({'customer_id': 2, 'window_end': 38, 'orders': 2, 'total_price': 400}, 1)
emit: ({'customer_id': 1, 'window_end': 35, 'orders': 3, 'total_price': 600}, 1)

=== Gap customer 2 abgelaufen → Emit ===
emit: ({'customer_id': 2, 'window_end': 80, 'orders': 1, 'total_price': 100}, 1)
emit: ({'customer_id': 2, 'window_end': 38, 'orders': 2, 'total_price': 400}, 0)
emit: ({'customer_id': 1, 'window_end': 35, 'orders': 3, 'total_price': 600}, 0)

=== Hard Reset bei max_session_int → neue session_bucket ===
emit: ({'customer_id': 1, 'window_end': 35, 'orders': 3, 'total_price': 600}, -1)
emit: ({'customer_id': 2, 'window_end': 38, 'orders': 2, 'total_price': 400}, -1)
emit: ({'customer_id': 2, 'window_end': 80, 'orders': 1, 'total_price': 100}, -1)

=== Gap neue Session customer 1 abgelaufen → Emit ===
emit: ({'customer_id': 1, 'window_end': 230, 'orders': 2, 'total_price':

In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

#

order_source_str = "orders"
sink_str = "orders_session_threshold"
#
gap_int = ts_step_int * 20
max_session_int = ts_step_int * 200
#
order_tn = (
    Tn.source(order_source_str)
    .expire(lambda x: x["value"]["ts"],
            lambda x: (x // max_session_int) * max_session_int + max_session_int)
    .map(lambda x: {"customer_id": x["value"]["customer_id"],
                    "price": x["value"]["price"],
                    "ts": x["value"]["ts"]})
    .distinct()
)
#
group_by_tn = (
    order_tn
    .group_by_agg(
        by_function=lambda x: (
            x["customer_id"],
            (x["ts"] // max_session_int) * max_session_int
        ),
        select_function=lambda x: x,
        projection_function=lambda by, agg: {
            "customer_id": by[0],
            "session_bucket": by[1],
            "window_end": agg["last_ts"] + gap_int,
            "orders": agg["orders"],
            "total_price": agg["total_price"],
            "last_ts": agg["last_ts"]
        },
        agg_function=lambda agg, x, w: {
            "orders": agg["orders"] + w,
            "total_price": agg["total_price"] + x["price"] * w,
            "last_ts": max(agg["last_ts"], x["ts"]) if w > 0 else agg["last_ts"]
        },
        agg_initial_any={"orders": 0, "total_price": 0, "last_ts": 0}
    )
)._peek("group")
#
window_tn = (
    group_by_tn
    .join(
        order_tn
        .map(lambda x: x["ts"])
        .max(lambda x: x, lambda x: {"ts": x}),
        lambda l, r: (r["ts"] > l["last_ts"] + gap_int
                      or l["total_price"] > 1000
                      or l["orders"] > 20),
        lambda l, _: l
    )
)._peek("expel")
#
built_tn = Tn.build(window_tn.sink(sink_str))



In [ ]:
built_tn.reset()

def push(customer_id, price, ts):
    built_tn.push(order_source_str, [{'value': {'customer_id': customer_id, 'price': price, 'ts': ts}}])
    built_tn.latest()

print("=== Session läuft, kein Emit ===")
push(1, 100, 0)   # customer 1, total=100, session_bucket=0
push(1, 200, 5)   # customer 1, total=300
push(1, 300, 10)  # customer 1, total=600

print("\n=== Threshold >1000 → Emit, aber Session läuft weiter ===")
push(1, 500, 15)  # customer 1, total=1100 → feuert! session_bucket=0 bleibt offen

print("\n=== Session läuft weiter im selben bucket ===")
push(1, 100, 20)  # customer 1, total=1200, last_ts=20

print("\n=== Gap abgelaufen → Emit mit kumulativem total ===")
push(2, 50, 60)   # watermark=60 > last_ts=20 + gap=20 → customer 1 feuert mit total=1200!

print("\n=== customer 2 threshold ===")
push(2, 400, 65)
push(2, 400, 70)
push(2, 400, 75)  # customer 2, total=1250 → feuert!

print("\n=== Hard Reset bei max_session_int ===")
push(1, 999, max_session_int)  # customer 1 neue session_bucket → frischer state

In [ ]:
import cloudpickle

built_tn.reset()
order_generator = OrderGenerator()

state_size_kb_float_list = []
for i in range(100):
    print(f"Step {i + 1}")
    order_message_dict_list = [order_generator.generate() for _ in range(100)]
    built_tn.push(order_source_str, order_message_dict_list)
    built_tn.latest()
    state_size_kb_float = len(cloudpickle.dumps(built_tn)) / 1024
    state_size_kb_float_list.append(state_size_kb_float)
#
print(f"Last 10 state sizes (KB): {state_size_kb_float_list[-10:]}")
